# 🎤 GPT-SoVITS 全自动训练 - 达妮娅语音

**数据集**: https://huggingface.co/datasets/huanx/daniya-voice-gptsovits

按顺序运行所有单元格，全自动完成训练。

**前提**: 右侧 Settings → Accelerator → **GPU T4**，Internet → **On**

## Step 1: 安装依赖

In [ ]:
!git clone https://github.com/RVC-Boss/GPT-SoVITS.git
%cd /kaggle/working/GPT-SoVITS

# 安装依赖
!pip install -r requirements.txt 2>&1 | tail -5
!pip install huggingface_hub 2>&1 | tail -3

print('✅ Step 1 完成')

## Step 2: 下载预训练模型

In [ ]:
import subprocess, zipfile, os
from huggingface_hub import hf_hub_download

# 下载预训练模型 (pretrained_models.zip)
print('下载 pretrained_models.zip ...')
path = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                        filename='pretrained_models.zip',
                        local_dir='.')

# 解压
print('解压中 ...')
with zipfile.ZipFile(path, 'r') as z:
    z.extractall('.')

# 检查
files = os.listdir('pretrained_models')
print(f'预训练模型: {len(files)} 个文件')
for f in files:
    print(f'  {f}')

# 下载 G2PW 模型
print('\n下载 G2PWModel.zip ...')
g2pw_path = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                             filename='G2PWModel.zip',
                             local_dir='.')
with zipfile.ZipFile(g2pw_path, 'r') as z:
    z.extractall('GPT_SoVITS/text/')

print('\n✅ Step 2 完成')

## Step 3: 下载数据集

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

# 下载数据集
print('下载达妮娅数据集 ...')
snapshot_download(repo_id='huanx/daniya-voice-gptsovits',
                   repo_type='dataset',
                   local_dir='/kaggle/working/daniya_dataset')

print('✅ Step 3 完成')

## Step 4: 准备数据目录

In [ ]:
import pandas as pd

os.chdir('/kaggle/working/GPT-SoVITS')

exp_name = 'daniya'
exp_dir = f'output/{exp_name}'

# 创建目录结构
dirs = [
    f'{exp_dir}/5-wav32k',
    f'{exp_dir}/1_16k_wavs',
    f'{exp_dir}/1_32k_wavs',
    f'{exp_dir}/2a-asr',
    f'{exp_dir}/2b-feature',
    f'{exp_dir}/3-bert',
    f'{exp_dir}/4-cnhubert',
    f'{exp_dir}/6-name2semantic',
    f'{exp_dir}/logs_s1',
    f'{exp_dir}/logs_s2_v2',
    f'{exp_dir}/weights',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

# 复制音频
audio_src = '/kaggle/working/daniya_dataset/audio'
wav_files = sorted(Path(audio_src).glob('*.wav'))
for f in wav_files:
    shutil.copy2(f, f'{exp_dir}/5-wav32k/')
    shutil.copy2(f, f'{exp_dir}/1_16k_wavs/')
    shutil.copy2(f, f'{exp_dir}/1_32k_wavs/')
print(f'复制 {len(wav_files)} 个音频')

# 生成 .list 文件
metadata = pd.read_csv('/kaggle/working/daniya_dataset/metadata.csv')
list_lines = []
for _, row in metadata.iterrows():
    wav_path = os.path.abspath(f'{exp_dir}/5-wav32k/{row["file"]}')
    text = str(row['text']).strip()
    list_lines.append(f'{wav_path}|daniya|zh|{text}')

inp_text_path = f'{exp_dir}/2a-asr/{exp_name}.list'
with open(inp_text_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(list_lines) + '\n')

print(f'生成 {len(list_lines)} 条训练记录')
print(f'示例: {list_lines[0][:80]}...')
print('✅ Step 4 完成')

## Step 5: 数据预处理

In [ ]:
import subprocess

os.chdir('/kaggle/working/GPT-SoVITS')

exp_name = 'daniya'
exp_dir = os.path.abspath(f'output/{exp_name}')
inp_text = os.path.abspath(f'{exp_dir}/2a-asr/{exp_name}.list')
inp_wav_dir = os.path.abspath(f'{exp_dir}/5-wav32k')

# 预训练模型路径
pretrained_dir = os.path.abspath('pretrained_models')
bert_dir = f'{pretrained_dir}/chinese-roberta-wwm-ext-large'
hubert_dir = f'{pretrained_dir}/chinese-hubert-base'

print(f'bert_dir: {bert_dir}')
print(f'hubert_dir: {hubert_dir}')
print(f'inp_text: {inp_text}')
print(f'inp_wav_dir: {inp_wav_dir}')

# 环境变量 (GPT-SoVITS 脚本通过环境变量读取参数)
base_env = {
    **os.environ,
    'inp_text': inp_text,
    'inp_wav_dir': inp_wav_dir,
    'exp_name': exp_name,
    'opt_dir': exp_dir,
    'i_part': '0',
    'all_parts': '1',
    'is_half': 'True',
}

# 5a: 提取文本特征
print('\n=== 5a: 提取文本特征 ===')
env1 = {**base_env, 'bert_pretrained_dir': bert_dir}
r = subprocess.run(['python', '-s', 'GPT_SoVITS/prepare_datasets/1-get-text.py'],
                    env=env1, capture_output=True, text=True)
print(r.stdout[-500:] if r.stdout else 'no stdout')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 1-get-text 完成')

# 5b: 提取 HuBERT 特征
print('\n=== 5b: 提取音频特征 ===')
env2 = {**base_env, 'cnhubert_base_dir': hubert_dir}
r = subprocess.run(['python', '-s', 'GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py'],
                    env=env2, capture_output=True, text=True)
print(r.stdout[-500:] if r.stdout else 'no stdout')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 2-get-hubert 完成')

# 5c: 提取语义
print('\n=== 5c: 提取语义 ===')
env3 = {**base_env, 'cnhubert_base_dir': hubert_dir}
r = subprocess.run(['python', '-s', 'GPT_SoVITS/prepare_datasets/3-get-semantic.py'],
                    env=env3, capture_output=True, text=True)
print(r.stdout[-500:] if r.stdout else 'no stdout')
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 3-get-semantic 完成')

# 检查产出
print('\n=== 产出检查 ===')
for subdir in ['2a-asr', '3-bert', '4-cnhubert', '5-wav32k', '6-name2semantic']:
    path = f'{exp_dir}/{subdir}'
    if os.path.exists(path):
        files = os.listdir(path)
        print(f'  {subdir}: {len(files)} 个文件')

print('\n✅ Step 5 完成')

## Step 6: 训练 SoVITS

In [ ]:
import json

os.chdir('/kaggle/working/GPT-SoVITS')

exp_name = 'daniya'
exp_dir = os.path.abspath(f'output/{exp_name}')

# 加载 SoVITS 配置模板
with open('GPT_SoVITS/configs/s2.json') as f:
    s2 = json.load(f)

# 修改训练参数
s2['train']['batch_size'] = 4
s2['train']['epochs'] = 200
s2['train']['save_every_epoch'] = 50
s2['train']['if_save_latest'] = True
s2['train']['if_save_every_weights'] = True
s2['train']['fp16_run'] = True
s2['train']['grad_ckpt'] = True
s2['data']['exp_dir'] = exp_dir
s2['s2_ckpt_dir'] = exp_dir
s2['name'] = exp_name
s2['version'] = 'v2'

tmp = '/tmp/s2_config.json'
with open(tmp, 'w') as f:
    json.dump(s2, f, indent=2)

print(f'SoVITS 训练: batch_size=4, epochs=200, save_every=50')
!python -s GPT_SoVITS/s2_train.py --config /tmp/s2_config.json

print('✅ Step 6 完成')

## Step 7: 训练 GPT

In [ ]:
import yaml

os.chdir('/kaggle/working/GPT-SoVITS')

exp_name = 'daniya'
exp_dir = os.path.abspath(f'output/{exp_name}')
s1_dir = f'{exp_dir}/logs_s1'

s1 = {
    'train_semantic_path': f'{s1_dir}/6-name2semantic.tsv',
    'train_phoneme_path': f'{s1_dir}/2-name2text.txt',
    'output_dir': s1_dir,
    'train': {
        'seed': 1234,
        'epochs': 100,
        'batch_size': 8,
        'save_every_n_epoch': 1,
        'if_save_latest': True,
        'if_save_every_weights': True,
        'half_weights_save_dir': f'{exp_dir}/weights',
        'exp_name': exp_name,
        'precision': '16-mixed',
        'gradient_clip': 1.0,
    },
    'optimizer': {
        'lr': 0.01,
        'lr_init': 0.00001,
        'lr_end': 0.0001,
        'warmup_steps': 2000,
        'decay_steps': 40000,
    },
    'data': {
        'max_eval_sample': 8,
        'max_sec': 54,
        'num_workers': 4,
        'pad_val': 1024,
    },
    'model': {
        'vocab_size': 1025,
        'phoneme_vocab_size': 512,
        'embedding_dim': 512,
        'hidden_dim': 512,
        'head': 16,
        'linear_units': 2048,
        'n_layer': 24,
        'dropout': 0,
        'EOS': 1024,
        'random_bert': 0,
    },
    'inference': {
        'top_k': 5,
    },
}

tmp = '/tmp/s1_config.yaml'
with open(tmp, 'w') as f:
    yaml.dump(s1, f, default_flow_style=False)

print(f'GPT 训练: batch_size=8, epochs=100, save_every=1')
!python -s GPT_SoVITS/s1_train.py --config_file /tmp/s1_config.yaml

print('✅ Step 7 完成')

## Step 8: 打包下载

In [ ]:
os.chdir('/kaggle/working/GPT-SoVITS')

print('=== 训练产出 ===')
for root, dirs, files in os.walk('output'):
    for f in sorted(files):
        if f.endswith(('.pth', '.ckpt')):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  {fpath} ({size_mb:.1f} MB)')

!tar -czf /kaggle/working/daniya_gptsovits_model.tar.gz output/
!ls -lh /kaggle/working/daniya_gptsovits_model.tar.gz

from IPython.display import FileLink, display
display(FileLink('/kaggle/working/daniya_gptsovits_model.tar.gz'))
print('\n✅ 完成! 点击链接下载模型')